# TitanAgent 実験1: 完全ゼロからの日本語 Titans (MAC) 事前学習

**事前学習済みモデルも外部トークナイザーも一切ダウンロードしない、完全ゼロ構成**です。
**Google Colab（有料枠 L4 / A100 / T4）** と **Kaggle（無料枠 T4 / P100・バックグラウンド最大12時間）** の両方でそのまま動きます。

### このノートブックの特徴
1. **外部モデル参照ゼロ**：`titans-pytorch` の `MemoryAsContextTransformer`（MAC 構造）をランダム初期値から構築
2. **自前学習の日本語 BPE トークナイザー（デフォルト・語彙4096）**：学習開始前にコーパスから数秒〜十数秒で ByteLevel BPE をゼロから学習（`<unk>` 発生ゼロ）。バイト単位（語彙257）より同じ計算量で約 4〜5 倍の日本語テキストを学習可能（`'byte'` への切替にも対応）
3. **チェックポイントからの再開対応**：`RESUME_FROM` を指定するだけで Kaggle の 12 時間枠を跨いだ継続学習が可能
4. **コーパス切り替え対応**：日本語コーパス（`hotchpotch/fineweb-2-edu-japanese`）のストリーミング学習と、完全オフラインの仮テキスト学習（`USE_DUMMY_TEXT = True`）を切り替え可能

In [ ]:
# 1. 環境確認 & ライブラリインストール
import os, sys, torch

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print(f'Environment: Colab={IN_COLAB}, Kaggle={IN_KAGGLE}')
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip -q install -U titans-pytorch datasets tokenizers

In [ ]:
# 2. 保存先ディレクトリ & 再開設定
USE_GOOGLE_DRIVE = False  # Colabで学習結果をGoogle Driveに保存したい場合は True

# 再開設定:
#   ''       : 毎回ゼロから学習
#   'auto'   : OUTPUT_DIR/latest.pt があれば自動再開
#   パス指定  : 例 '/kaggle/input/my-prev-run/checkpoints/exp1_bpe/latest.pt'
RESUME_FROM = 'auto'

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/TitanAgent/checkpoints/exp1_small_bpe'
elif IN_KAGGLE:
    OUTPUT_DIR = '/kaggle/working/checkpoints/exp1_small_bpe'
else:
    OUTPUT_DIR = './checkpoints/exp1_small_bpe'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR, '| RESUME_FROM =', repr(RESUME_FROM))

In [ ]:
# 3. 実験設定
PRESET = 'small'          # 'tiny' (~12M), 'small' (~25M), 'medium' (~50M)
TOKENIZER_TYPE = 'bpe'    # 'bpe' (自前学習の日本語BPE・推奨) or 'byte' (UTF-8バイト単位)
VOCAB_SIZE = 4096         # TOKENIZER_TYPE='bpe' の目標語彙サイズ (4096〜8192推奨)
TOKENIZER_TRAIN_DOCS = 10000  # BPE学習に使うドキュメント数 (約10〜20秒で完了)

MAX_STEPS = 3000          # small(32,768 tok/step): 3000 step ≈ 98M tokens (Kaggle T4で9〜10h目安)
WARMUP_STEPS = 200
LOG_EVERY = 20
SAMPLE_EVERY = 200
SAVE_EVERY = 500

# True にすると外部通信ゼロの内蔵仮テキストで即座に動作確認できます
# False にすると Hugging Face の日本語コーパスをストリーミング学習します
USE_DUMMY_TEXT = False
DATASET_NAME = 'hotchpotch/fineweb-2-edu-japanese'
DATASET_CONFIG = 'sample_10BT'
SEED = 42

PRESETS = {
    'tiny':   dict(dim=256, depth=4, segment_len=64,  num_persist_mem_tokens=4, num_longterm_mem_tokens=16, seq_len=512,  batch_size=8, grad_accum=4, lr=8e-4),
    'small':  dict(dim=384, depth=6, segment_len=128, num_persist_mem_tokens=8, num_longterm_mem_tokens=32, seq_len=1024, batch_size=4, grad_accum=8, lr=5e-4),
    'medium': dict(dim=512, depth=8, segment_len=128, num_persist_mem_tokens=8, num_longterm_mem_tokens=32, seq_len=1024, batch_size=4, grad_accum=8, lr=3e-4),
}
cfg = PRESETS[PRESET]
print('Selected config:', cfg, 'TOKENIZER_TYPE =', TOKENIZER_TYPE, 'USE_DUMMY_TEXT =', USE_DUMMY_TEXT)

In [ ]:
# 4. 完全ゼロ構築：自前トークナイザー & Titans (MAC) モデル
import math, time
from pathlib import Path
import torch
from titans_pytorch import MemoryAsContextTransformer

torch.manual_seed(SEED)

class ByteTokenizer:
    tokenizer_type = 'byte'
    vocab_size = 257
    eos_token_id = 256

    @staticmethod
    def encode(text: str):
        return list(text.encode('utf-8'))

    @staticmethod
    def decode(ids):
        return bytes(i for i in ids if 0 <= i < 256).decode('utf-8', errors='ignore')

class ScratchBPETokenizer:
    tokenizer_type = 'bpe'

    def __init__(self, tok):
        self._tok = tok
        self.eos_token_id = int(tok.token_to_id('<|eos|>'))
        self.vocab_size = int(tok.get_vocab_size())

    def encode(self, text: str):
        return self._tok.encode(text).ids

    def decode(self, ids):
        filtered = [i for i in ids if 0 <= i < self.vocab_size and i != self.eos_token_id]
        return self._tok.decode(filtered, skip_special_tokens=True)

    def save(self, path: Path):
        path.parent.mkdir(parents=True, exist_ok=True)
        self._tok.save(str(path))

    @classmethod
    def load(cls, path: Path):
        from tokenizers import Tokenizer
        return cls(Tokenizer.from_file(str(path)))

    @classmethod
    def train_from_texts(cls, texts, vocab_size=4096, min_frequency=2):
        from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
        tok = Tokenizer(models.BPE())
        tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        tok.decoder = decoders.ByteLevel()
        trainer = trainers.BpeTrainer(
            vocab_size=vocab_size,
            min_frequency=min_frequency,
            special_tokens=['<|eos|>', '<|pad|>'],
            initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        )
        tok.train_from_iterator(texts, trainer=trainer)
        return cls(tok)

DUMMY_CORPUS_JA = [
    '人工知能の長期記憶モジュールは、予測しにくい新しい情報ほど強く記憶し、重要度の低い情報を忘却する仕組みを持っています。',
    'ユーザー: 昨日の作業の続きをお願い。\n思考: 前回のログを確認すると、実験1の事前学習スクリプトを作成していた。まずは設定を確認しよう。\n回答: はい、実験1の続きから進めます。',
    '新しいチャットを作らずに1本の会話で継続する場合、会話履歴だけでなく思考プロセスや実行結果も連続した文脈として保持することが重要になります。',
    'ユーザー: 回答はなるべく簡潔な箇条書きにしてほしい。\n思考: ユーザーは簡潔な箇条書きを好む。今後の応答でもこの指示パターンを維持する。\n回答: 了解しました。今後は簡潔な箇条書きでお答えします。',
    'テスト時学習では、推論しながらニューラル記憶の重みを更新し続けることで、極端に長い文脈にも対応できます。',
]

if torch.cuda.is_available():
    device = torch.device('cuda')
    major, _ = torch.cuda.get_device_capability()
    if major >= 8 and torch.cuda.is_bf16_supported():
        amp_dtype = torch.bfloat16
    elif major >= 7:
        amp_dtype = torch.float16
    else:
        amp_dtype = torch.float32
else:
    device = torch.device('cpu')
    amp_dtype = torch.float32

use_amp = device.type == 'cuda' and amp_dtype in (torch.bfloat16, torch.float16)
use_scaler = device.type == 'cuda' and amp_dtype == torch.float16
print(f'device={device}, amp_dtype={amp_dtype}')

output_dir = Path(OUTPUT_DIR)
resume_path = None
if RESUME_FROM == 'auto':
    cand = output_dir / 'latest.pt'
    if cand.exists():
        resume_path = cand
elif RESUME_FROM:
    cand = Path(RESUME_FROM)
    if cand.exists():
        resume_path = cand
    else:
        print(f'[Warning] RESUME_FROM not found: {cand} (starting fresh)')

def iter_tokenizer_texts(dataset_name, dataset_config, use_dummy, max_docs, seed=42):
    if use_dummy:
        for _ in range(20):
            for t in DUMMY_CORPUS_JA:
                yield t
        return
    from datasets import load_dataset
    ds = load_dataset(dataset_name, name=dataset_config, split='train', streaming=True)
    ds = ds.shuffle(seed=seed, buffer_size=1000)
    count = 0
    for item in ds:
        text = item.get('text', '')
        if not text:
            continue
        yield text
        count += 1
        if count >= max_docs:
            break

if TOKENIZER_TYPE == 'byte':
    tokenizer = ByteTokenizer()
else:
    tok_candidates = [output_dir / 'tokenizer.json']
    if resume_path is not None:
        tok_candidates.append(resume_path.parent / 'tokenizer.json')
    loaded_tok = None
    for cand in tok_candidates:
        if cand.exists():
            loaded_tok = ScratchBPETokenizer.load(cand)
            if cand != output_dir / 'tokenizer.json':
                loaded_tok.save(output_dir / 'tokenizer.json')
            print(f'[Tokenizer] Loaded existing BPE from {cand} (vocab={loaded_tok.vocab_size})')
            break
    if loaded_tok is None:
        t_tok = time.time()
        print(f'[Tokenizer] Training from-scratch Japanese BPE (vocab={VOCAB_SIZE}, docs={TOKENIZER_TRAIN_DOCS if not USE_DUMMY_TEXT else "dummy"})...')
        texts = iter_tokenizer_texts(DATASET_NAME, DATASET_CONFIG, USE_DUMMY_TEXT, TOKENIZER_TRAIN_DOCS, seed=SEED)
        loaded_tok = ScratchBPETokenizer.train_from_texts(
            texts, vocab_size=VOCAB_SIZE, min_frequency=1 if USE_DUMMY_TEXT else 2
        )
        loaded_tok.save(output_dir / 'tokenizer.json')
        print(f'[Tokenizer] Trained & saved (vocab={loaded_tok.vocab_size}, {time.time() - t_tok:.1f}s)')
    tokenizer = loaded_tok

# 圧縮率の確認
_sample = DUMMY_CORPUS_JA[0]
print(f'[Tokenizer check] chars={len(_sample)}, utf8_bytes={len(_sample.encode("utf-8"))}, tokens={len(tokenizer.encode(_sample))}')

model_cfg = dict(
    num_tokens=tokenizer.vocab_size,
    dim=cfg['dim'],
    depth=cfg['depth'],
    segment_len=cfg['segment_len'],
    num_persist_mem_tokens=cfg['num_persist_mem_tokens'],
    num_longterm_mem_tokens=cfg['num_longterm_mem_tokens'],
)

ckpt = None
start_step = 0
if resume_path is not None:
    ckpt = torch.load(resume_path, map_location=device)
    if 'model_cfg' in ckpt:
        model_cfg = ckpt['model_cfg']
    start_step = int(ckpt['step'])
    print(f'[Resume] Loaded checkpoint {resume_path} at step {start_step}')

model = MemoryAsContextTransformer(**model_cfg).to(device)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M (vocab={model_cfg["num_tokens"]})')

def token_stream(dataset_name, dataset_config, tokenizer, seq_len, use_dummy=False, seed=42):
    chunk_len = seq_len + 1
    buf = []
    if use_dummy:
        idx = 0
        while True:
            text = DUMMY_CORPUS_JA[idx % len(DUMMY_CORPUS_JA)]
            idx += 1
            buf.extend(tokenizer.encode(text))
            buf.append(tokenizer.eos_token_id)
            while len(buf) >= chunk_len:
                yield buf[:chunk_len]
                buf = buf[chunk_len:]

    from datasets import load_dataset
    epoch = 0
    while True:
        ds = load_dataset(dataset_name, name=dataset_config, split='train', streaming=True)
        ds = ds.shuffle(seed=seed + epoch, buffer_size=1000)
        for item in ds:
            text = item.get('text', '')
            if not text:
                continue
            ids = tokenizer.encode(text)
            if not ids:
                continue
            buf.extend(ids)
            buf.append(tokenizer.eos_token_id)
            while len(buf) >= chunk_len:
                yield buf[:chunk_len]
                buf = buf[chunk_len:]
        epoch += 1

def batch_iter(stream, batch_size):
    while True:
        yield torch.tensor([next(stream) for _ in range(batch_size)], dtype=torch.long)

def cosine_lr(step, warmup, total, max_lr, min_ratio=0.1):
    if step < warmup:
        return max_lr * (step + 1) / max(1, warmup)
    p = min(max((step - warmup) / max(1, total - warmup), 0.0), 1.0)
    return max_lr * (min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * p)))

@torch.no_grad()
def generate_sample(prompt, max_new_tokens=80):
    model.eval()
    ids = tokenizer.encode(prompt) or [tokenizer.eos_token_id]
    x = torch.tensor([ids], dtype=torch.long, device=device)
    # titans-pytorch の model.sample(prompt, seq_len) はプロンプト込みの合計長を指定する
    out = model.sample(x, len(ids) + max_new_tokens)[0].tolist()
    model.train()
    return tokenizer.decode(out)

In [ ]:
# 5. 学習ループ実行
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], betas=(0.9, 0.95), weight_decay=0.1)
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
loss_history = []

if ckpt is not None:
    model.load_state_dict(ckpt['model_state_dict'])
    if 'optimizer_state_dict' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if use_scaler and 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    loss_history = list(ckpt.get('loss_history', []))

stream = token_stream(
    DATASET_NAME, DATASET_CONFIG, tokenizer, cfg['seq_len'],
    use_dummy=USE_DUMMY_TEXT, seed=SEED + start_step
)
batches = batch_iter(stream, cfg['batch_size'])

tokens_per_step = cfg['batch_size'] * cfg['grad_accum'] * cfg['seq_len']
print(f'Training step {start_step} -> {MAX_STEPS} ({tokens_per_step:,} tokens/step, total ~{MAX_STEPS * tokens_per_step / 1e6:.2f}M tokens)')

sample_prompts = [
    '人工知能の長期記憶において重要なのは、',
    'ユーザー: 昨日の作業の続きをお願い。\n思考: ',
]

model.train()
t0 = time.time()

for step in range(start_step + 1, MAX_STEPS + 1):
    lr = cosine_lr(step - 1, WARMUP_STEPS, MAX_STEPS, cfg['lr'])
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0

    for _ in range(cfg['grad_accum']):
        x = next(batches).to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            loss = model(x, return_loss=True)
            scaled = loss / cfg['grad_accum']
        if use_scaler:
            scaler.scale(scaled).backward()
        else:
            scaled.backward()
        accum_loss += float(loss.item()) / cfg['grad_accum']

    if use_scaler:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    loss_history.append((step, accum_loss))

    if step % LOG_EVERY == 0 or step == start_step + 1:
        elapsed = time.time() - t0
        steps_done = step - start_step
        tok_s = (steps_done * tokens_per_step) / max(elapsed, 1e-6)
        if tokenizer.tokenizer_type == 'byte':
            metric = f'bpb {accum_loss / math.log(2):.3f}'
        else:
            metric = f'ppl {math.exp(min(accum_loss, 20.0)):.2f}'
        print(f'step {step:6d}/{MAX_STEPS} | loss {accum_loss:.4f} | {metric} | lr {lr:.2e} | {tok_s:,.0f} tok/s')

    if step % SAMPLE_EVERY == 0:
        print('-' * 60)
        for p in sample_prompts:
            print(f'[Prompt] {p}\n[Output] {generate_sample(p, 80)}\n')
        print('-' * 60)

    if step % SAVE_EVERY == 0 or step == MAX_STEPS:
        payload = {
            'step': step,
            'loss': accum_loss,
            'model_cfg': model_cfg,
            'tokenizer_type': tokenizer.tokenizer_type,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss_history': loss_history,
        }
        ckpt_path = output_dir / f'checkpoint_step_{step:06d}.pt'
        torch.save(payload, ckpt_path)
        torch.save(payload, output_dir / 'latest.pt')
        print(f'[Checkpoint] Saved: {ckpt_path} (and latest.pt)')

In [ ]:
# 6. 簡易テスト：複数セグメント跨ぎの生成確認
long_prompt = (
    '設定メモ: ユーザーのプロジェクト名は TitanAgent で、好みの回答スタイルは簡潔な箇条書きです。\n'
    + 'ログ: データの読み込みと前処理を実行しました。特にエラーは発生していません。\n' * 10
    + 'ユーザー: 今のプロジェクト名と、好みの回答スタイルを教えて。\n回答: '
)
print('Prompt tokens:', len(tokenizer.encode(long_prompt)), '| bytes:', len(long_prompt.encode('utf-8')))
print('[Output]', generate_sample(long_prompt, max_new_tokens=80))